<a href="https://colab.research.google.com/github/himaja-56/VRSU/blob/main/VRSU_TransferLearningProgram7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import cv2
import gc

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from google.colab import files

tf.keras.backend.clear_session()
gc.collect ()

#--------- Load Dataset (only for training) ---------
(ds_train, ds_test), ds_info = tfds.load(
  "oxford_iiit_pet",
  split=["train[:80%]", "train[80%:]"],
  as_supervised=True,
  with_info=True
)
NUM_CLASSES = ds_info.features["label"]. num_classes
IMG_SIZE = 128
BATCH_SIZE = 16

def preprocess (image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = preprocess_input (image)
    return image, label

train_ds = ds_train.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = ds_test.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# --------- Load MobileNetv2 (Pretrained) ---------
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False


x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(64, activation="relu")(x)
output = Dense (NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(train_ds, epochs=3)

loss, acc = model.evaluate(test_ds)
print("Validation Accuracy:", acc)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_iiit_pet/incomplete.KC6J69_4.0.0/oxford_iiit_pet-train.tfrecord*...…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_iiit_pet/incomplete.KC6J69_4.0.0/oxford_iiit_pet-test.tfrecord*...:…

Dataset oxford_iiit_pet downloaded and prepared to /root/tensorflow_datasets/oxford_iiit_pet/4.0.0. Subsequent calls will reuse this data.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/3
184/184 ━━━━━━━━━━━━━━━━━━━━ 26s 120ms/step - accuracy: 0.6607 - loss: 1.2317
Epoch 2/3
184/184 ━━━━━━━━━━━━━━━━━━━━ 23s 122ms/step - accuracy: 0.9151 - loss: 0.2893
Epoch 3/3
184/184 ━━━━━━━━━━━━━━━━━━━━ 22s 116ms/step - accuracy: 0.9749 - loss: 0.1216
46/46 ━━━━━━━━━━━━━━━━━━━━ 7s 112ms/step - accuracy: 0.8438 - loss: 0.4786
Validation Accuracy: 0.84375
